# Nivel 2: Procesamiento de archivos

**Autor:** Sebastian Muñoz Palacio  
**Curso:** Big Data Analytics  
**Programa:** Maestría en Ciencia de Datos - Universidad Pontificia Bolivariana (UPB)

Ahora los datos no están escritos en el código: se leen de archivos de texto en la carpeta `data/`.

- Para **un archivo**, cada línea es un dato de entrada.
- Para **varios archivos**, cada dato es una tupla `(nombre_archivo, línea)`, así el mapper sabe de qué archivo viene cada línea.

In [1]:
from pathlib import Path
from mapreduce_framework import mapreduce, print_results

DATA_DIR = Path("data")


def read_lines(path):
    """Lee un archivo y devuelve sus líneas no vacías."""
    with open(path, encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def read_dir(directory):
    """Lee todos los .txt de una carpeta y devuelve tuplas (nombre_archivo, línea)."""
    data = []
    for path in sorted(Path(directory).glob("*.txt")):
        for line in read_lines(path):
            data.append((path.name, line))
    return data


print("Archivos en data/:", [p.name for p in sorted(DATA_DIR.glob("*.txt"))])

Archivos en data/: ['sample_bigdata.txt', 'sample_text.txt']


---
## Ejercicio 2.1: Distribución por longitud de palabra

**Objetivo:** contar cuántas palabras hay de cada longitud en `sample_text.txt`.

- **mapper** → emite `(longitud, 1)` por cada palabra.
- **reducer** → suma.

In [2]:
lines = read_lines(DATA_DIR / "sample_text.txt")


def mapper(line):
    """Emite (longitud_de_palabra, 1) por cada palabra."""
    for word in line.split():
        word = word.strip('.,!?')
        if word:                        # ignora "palabras" que eran solo signos
            yield (len(word), 1)


def reducer(length, counts):
    """Cuenta cuántas palabras tienen esa longitud."""
    return sum(counts)


results = mapreduce(lines, mapper, reducer)
for length in sorted(results):          # ordenado por longitud
    print(f"{length:2}: {results[length]} words")

print("Datos vacíos:", mapreduce([], mapper, reducer))

 1: 10 words
 2: 20 words
 3: 33 words
 4: 17 words
 5: 14 words
 6: 13 words
 7: 12 words
 8: 17 words
 9: 8 words
10: 12 words
11: 8 words
12: 7 words
13: 3 words
14: 1 words
Datos vacíos: {}


---
## Ejercicio 2.2: Palabras únicas por archivo

**Objetivo:** saber cuántas palabras distintas tiene cada archivo de `data/`.

- **mapper** → emite `(archivo, palabra)`.
- **reducer** → `len(set(palabras))`: el `set` elimina repetidas.

In [3]:
files_data = read_dir(DATA_DIR)


def mapper(file_data):
    """Emite (archivo, palabra) por cada palabra de la línea."""
    filename, line = file_data
    for word in line.lower().split():
        word = word.strip('.,!?')
        if word:
            yield (filename, word)


def reducer(filename, words):
    """Cuenta las palabras distintas del archivo."""
    return len(set(words))


results = mapreduce(files_data, mapper, reducer)
for filename, unique in results.items():
    print(f"{filename}: {unique} unique words")

print("Datos vacíos:", mapreduce([], mapper, reducer))

sample_bigdata.txt: 79 unique words
sample_text.txt: 116 unique words
Datos vacíos: {}


---
## Ejercicio 2.3: Índice invertido

**Objetivo:** para cada palabra, listar en qué archivos aparece. Así funcionan internamente los buscadores.

- **mapper** → emite `(palabra, archivo)`. Es el inverso del 2.2: ahora la clave es la palabra.
- **reducer** → lista ordenada de archivos sin repetir.

In [4]:
def mapper(file_data):
    """Emite (palabra, archivo) por cada palabra de la línea."""
    filename, line = file_data
    for word in line.lower().split():
        word = word.strip('.,!?')
        if word:
            yield (word, filename)


def reducer(word, filenames):
    """Devuelve la lista de archivos (sin repetir) donde aparece la palabra."""
    return sorted(set(filenames))


index = mapreduce(files_data, mapper, reducer)
print(f"Palabras indexadas: {len(index)}\n")

# Las palabras del ejemplo del enunciado
for word in ["mapreduce", "hadoop", "data"]:
    print(f"{word}: {index.get(word, [])}")

# Palabras que aparecen en todos los archivos
in_all = sorted(w for w, files in index.items() if len(files) == len(set(f for f, _ in files_data)))
print(f"\nPalabras presentes en todos los archivos ({len(in_all)}):")
print(", ".join(in_all))

print("\nDatos vacíos:", mapreduce([], mapper, reducer))

Palabras indexadas: 177

mapreduce: ['sample_text.txt']
hadoop: ['sample_bigdata.txt']
data: ['sample_bigdata.txt', 'sample_text.txt']

Palabras presentes en todos los archivos (18):
and, be, big, common, data, has, in, include, is, large, machine, of, processing, sets, that, the, to, variety

Datos vacíos: {}
